# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

My lane is still **Refresh / Content Opportunity Scoring**.

**Five plain-words contract answers:**

1. **What one raw row means:** In `fact_content_daily_performance`, one raw row means one `report_date` x one pseudonymized client x one pseudonymized content item. In plain words: one page-day.
2. **Which table I will use:** For this Week 3 slice I will use `fact_content_daily_performance` from the Hugging Face warehouse. Later I may join `dim_content` for page metadata, but this notebook keeps the contract small and uses the daily fact table only.
3. **Which time window:** I will use the mid-panel month `2026-03`, not the final June sample. Within March, my five features are built from `2026-03-01` through `2026-03-15`; the starter proxy label checks whether impressions fall in `2026-03-16` through `2026-03-31`.
4. **What I would predict or rank:** I would rank content items for human refresh/opportunity review. The temporary proxy label is `future_decline_proxy`: among pages with at least 100 first-half impressions, did second-half March impressions fall below 80% of first-half March impressions?
5. **One thing I deliberately exclude:** I exclude any label-derived column, including the deliberate `leaky_decline_hint` created below. I also keep `client_hash_id` and `content_hash_id` as context only, not model features.

**Lane feature-frame unit:** After aggregating the daily fact rows, one row in my lane slice means one pseudonymized content item for March 2026, with first-half-March features and a second-half-March proxy outcome.


In [1]:
import os
import getpass

import duckdb
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit


def get_hf_token():
    """Read token safely from env or Colab Secrets; never hard-code it in the notebook."""
    token = os.environ.get("HF_TOKEN")
    if token:
        return token
    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    return getpass.getpass("Paste your Hugging Face READ token (hf_...): ")


HF_TOKEN = get_hf_token()
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH_FACT = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

print("Connected to the March 2026 warehouse partition without storing the token in the notebook.")


Connected to the March 2026 warehouse partition without storing the token in the notebook.


## 2. Fields: feature / label / context / excluded

| Bucket | Fields | Why |
|---|---|---|
| Context | `client_hash_id`, `content_hash_id`, `report_date` | Needed for grouping, joining, splitting, and reading the table grain. IDs are not model features. |
| Five features | `gsc_impressions_first15`, `gsc_clicks_first15`, `gsc_avg_position_first15`, `ga4_sessions_first15`, `ga4_engagement_rate_first15` | These are observable in the first half of March before the proxy outcome window. |
| Label / proxy | `future_decline_proxy` | The temporary target: second-half March impressions are less than 80% of first-half March impressions for pages with at least 100 first-half impressions. |
| Excluded | `leaky_decline_hint`, target-window metrics as features, raw/private fields, IDs as features | These either reveal the answer, occur after the decision moment, are private, or are meaningless pseudonymized identifiers. |

**Available-when lines for the five features:**

- `gsc_impressions_first15`: knowable at the decision moment because it is summed only from `2026-03-01` through `2026-03-15`.
- `gsc_clicks_first15`: knowable at the decision moment because it is summed only from the first-half March feature window.
- `gsc_avg_position_first15`: knowable at the decision moment because it averages observed GSC position only inside the first-half March feature window.
- `ga4_sessions_first15`: knowable at the decision moment because it uses GA4 sessions from the first-half March feature window and only rows with `ga4_data_available IS TRUE`.
- `ga4_engagement_rate_first15`: knowable at the decision moment because it is derived from first-half March engaged sessions and sessions, not from the outcome window.


In [2]:
field_contract = pd.DataFrame(
    [
        {"field": "client_hash_id", "bucket": "context", "available_when": "Used for grouping/splitting only; not a feature."},
        {"field": "content_hash_id", "bucket": "context", "available_when": "Identifies the content item; not a feature."},
        {"field": "gsc_impressions_first15", "bucket": "feature", "available_when": "Known from 2026-03-01 through 2026-03-15."},
        {"field": "gsc_clicks_first15", "bucket": "feature", "available_when": "Known from 2026-03-01 through 2026-03-15."},
        {"field": "gsc_avg_position_first15", "bucket": "feature", "available_when": "Known from first-half March rows with real position data."},
        {"field": "ga4_sessions_first15", "bucket": "feature", "available_when": "Known from first-half March rows where GA4 data is available."},
        {"field": "ga4_engagement_rate_first15", "bucket": "feature", "available_when": "Computed from first-half March GA4 sessions and engaged sessions."},
        {"field": "future_decline_proxy", "bucket": "label / proxy", "available_when": "Defined from 2026-03-16 through 2026-03-31; never a feature."},
        {"field": "leaky_decline_hint", "bucket": "excluded", "available_when": "Created only to demonstrate leakage, then removed."},
    ]
)

field_contract


,field,bucket,available_when
0,client_hash_id,context,Used for grouping/splitting only; not a feature.
1,content_hash_id,context,Identifies the content item; not a feature.
2,gsc_impressions_first15,feature,Known from 2026-03-01 through 2026-03-15.
3,gsc_clicks_first15,feature,Known from 2026-03-01 through 2026-03-15.
4,gsc_avg_position_first15,feature,Known from first-half March rows with real pos...
5,ga4_sessions_first15,feature,Known from first-half March rows where GA4 dat...
6,ga4_engagement_rate_first15,feature,Computed from first-half March GA4 sessions an...
7,future_decline_proxy,label / proxy,Defined from 2026-03-16 through 2026-03-31; ne...
8,leaky_decline_hint,excluded,"Created only to demonstrate leakage, then remo..."


## 3. Verify it with queries (grain, counts, missing values, windows)

Below are the **exactly three verification queries** for the March 2026 warehouse slice.

1. **Grain query:** verifies that `report_date + client_hash_id + content_hash_id` has no duplicate keys.
2. **Count/date-span query:** counts March rows, clients, content items, and the min/max dates.
3. **Availability query:** uses `IS TRUE` filters to show how many rows survive GSC and GA4 availability checks.

After those checks, I build a five-feature page-month frame from the same March partition and run the deliberate leakage trap.


In [3]:
# Verification query 1: grain check.
q1_grain = con.sql(
    f"""
    WITH duplicate_keys AS (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS row_count
        FROM read_parquet('{MARCH_FACT}')
        GROUP BY 1, 2, 3
        HAVING COUNT(*) > 1
    )
    SELECT COUNT(*) AS duplicate_grain_keys
    FROM duplicate_keys
    """
).df()

# Verification query 2: row count and date span.
q2_count_span = con.sql(
    f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(DISTINCT client_hash_id) AS clients,
        COUNT(DISTINCT content_hash_id) AS content_items,
        MIN(report_date) AS min_report_date,
        MAX(report_date) AS max_report_date
    FROM read_parquet('{MARCH_FACT}')
    """
).df()

# Verification query 3: availability, explicitly checked with IS TRUE.
q3_availability = con.sql(
    f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_with_gsc_available,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS rows_with_ga4_available,
        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE
        ) AS rows_with_both_available
    FROM read_parquet('{MARCH_FACT}')
    """
).df()

print("Verification query 1: grain check")
print(q1_grain.to_string(index=False))
print("\nVerification query 2: row count and date span")
print(q2_count_span.to_string(index=False))
print("\nVerification query 3: availability checked with IS TRUE")
print(q3_availability.to_string(index=False))

# Build the five-feature frame from the same March slice.
feature_frame = con.sql(
    f"""
    WITH page_month AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS gsc_impressions_first15,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS gsc_clicks_first15,
            AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS gsc_avg_position_first15,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN ga4_sessions ELSE 0 END) AS ga4_sessions_first15,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN ga4_engaged_sessions ELSE 0 END) AS ga4_engaged_sessions_first15,
            SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS gsc_impressions_last16
        FROM read_parquet('{MARCH_FACT}')
        WHERE gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT
        client_hash_id,
        content_hash_id,
        gsc_impressions_first15,
        gsc_clicks_first15,
        gsc_avg_position_first15,
        ga4_sessions_first15,
        CASE
            WHEN ga4_sessions_first15 > 0
            THEN ga4_engaged_sessions_first15 * 1.0 / ga4_sessions_first15
            ELSE 0
        END AS ga4_engagement_rate_first15,
        CASE
            WHEN gsc_impressions_first15 >= 100
             AND gsc_impressions_last16 < 0.8 * gsc_impressions_first15
            THEN 1
            ELSE 0
        END AS future_decline_proxy
    FROM page_month
    WHERE gsc_impressions_first15 >= 100
      AND gsc_avg_position_first15 IS NOT NULL
      AND ga4_sessions_first15 >= 1
    """
).df()

feature_cols = [
    "gsc_impressions_first15",
    "gsc_clicks_first15",
    "gsc_avg_position_first15",
    "ga4_sessions_first15",
    "ga4_engagement_rate_first15",
]

print("\nFive-feature frame shape:", feature_frame.shape)
print("\nFive-feature frame sample:")
print(feature_frame.head(8).to_string(index=False))

# The trap: add one label-derived column on purpose, observe the suspicious jump, then remove it.
model_df = feature_frame.dropna(subset=feature_cols + ["future_decline_proxy"]).copy()
model_df = model_df.sample(n=min(40000, len(model_df)), random_state=42)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(
    splitter.split(model_df, model_df["future_decline_proxy"], groups=model_df["client_hash_id"])
)
train = model_df.iloc[train_idx].copy()
test = model_df.iloc[test_idx].copy()

honest_model = RandomForestClassifier(n_estimators=80, max_depth=5, min_samples_leaf=25, random_state=42)
honest_model.fit(train[feature_cols], train["future_decline_proxy"])
honest_scores = honest_model.predict_proba(test[feature_cols])[:, 1]

train["leaky_decline_hint"] = train["future_decline_proxy"]
test["leaky_decline_hint"] = test["future_decline_proxy"]
leaky_cols = feature_cols + ["leaky_decline_hint"]
leaky_model = RandomForestClassifier(n_estimators=80, max_depth=5, min_samples_leaf=25, random_state=42)
leaky_model.fit(train[leaky_cols], train["future_decline_proxy"])
leaky_scores = leaky_model.predict_proba(test[leaky_cols])[:, 1]

leakage_demo = pd.DataFrame(
    [
        {
            "run": "honest_features_only",
            "roc_auc": round(roc_auc_score(test["future_decline_proxy"], honest_scores), 3),
            "average_precision": round(average_precision_score(test["future_decline_proxy"], honest_scores), 3),
            "kept_for_modeling": True,
        },
        {
            "run": "with_deliberate_leak",
            "roc_auc": round(roc_auc_score(test["future_decline_proxy"], leaky_scores), 3),
            "average_precision": round(average_precision_score(test["future_decline_proxy"], leaky_scores), 3),
            "kept_for_modeling": False,
        },
    ]
)

print("\nLeakage demo: the leaky label-derived feature jumps toward perfect, then is removed.")
print(leakage_demo.to_string(index=False))


Verification query 1: grain check
 duplicate_grain_keys
                    0

Verification query 2: row count and date span
 march_rows  clients  content_items min_report_date max_report_date
    9841378       55         331437      2026-03-01      2026-03-31

Verification query 3: availability checked with IS TRUE
 total_rows  rows_with_gsc_available  rows_with_ga4_available  rows_with_both_available
    9841378                  3611061                   413966                    364347

Five-feature frame shape: (18010, 8)

Five-feature frame sample:
         client_hash_id          content_hash_id  gsc_impressions_first15  gsc_clicks_first15  gsc_avg_position_first15  ga4_sessions_first15  ga4_engagement_rate_first15  future_decline_proxy
client_20259bd6705d81d4 content_4d1f9fef25655a3e                   3197.0                22.0                  5.510027                  20.0                     0.000000                     0
client_20259bd6705d81d4 content_8e86b6d7b4ae673e      

## 4. Data limits

One named limitation of this slice: **it uses one mid-panel month only**. March 2026 is useful for query mechanics and a first contract, but one month cannot prove that a refresh recommendation will work across seasons, clients, or longer content lifecycles.

Other limits I need to keep in view:

- GA4 availability is much thinner than the raw March row count. The availability query shows that only a smaller subset survives `ga4_data_available IS TRUE`, so engagement features describe the measurable subset, not every page.
- The proxy label is a short-window decline signal, not proof that editing a page would cause recovery.
- Client histories are unbalanced, so later validation should use client-aware or time-aware splits.
- Pseudonymized IDs are allowed for grouping and splitting, but not as model features or public examples.


In [4]:
limitation_summary = pd.DataFrame(
    [
        {
            "limit": "one_month_slice",
            "why_it_matters": "March 2026 may not represent seasonality or all client histories.",
        },
        {
            "limit": "ga4_availability_filter",
            "why_it_matters": "Rows without GA4 availability cannot honestly support engagement features.",
        },
        {
            "limit": "proxy_not_causal",
            "why_it_matters": "A decline proxy does not prove a refresh would cause recovery.",
        },
    ]
)

limitation_summary


,limit,why_it_matters
0,one_month_slice,March 2026 may not represent seasonality or al...
1,ga4_availability_filter,Rows without GA4 availability cannot honestly ...
2,proxy_not_causal,A decline proxy does not prove a refresh would...


## Self-check

Before submitting, I checked each line honestly:

- [✓] Five plain-words contract answers are filled.
- [✓] The notebook uses the March 2026 mid-panel warehouse month, not the final June sample.
- [✓] Exactly three verification queries are shown with outputs: grain, count/date span, and availability.
- [✓] Availability is checked with `IS TRUE`.
- [✓] The feature frame has five features max.
- [✓] Each feature has an available-when explanation.
- [✓] The deliberate leakage trap is shown: the leaky score jumps toward perfect, then the leaky column is excluded.
- [✓] One limitation of the slice is named.
- [✓] No token, client name, raw URL, raw query, or private field is written into the notebook.
